# Parte 3 · Flexibilidad y optimismo (3.1–3.3)

**Responsabilidad: Jorge (etiqueta de tarea) · Grupo Taiwan.**

**Asistencia de IA:** OpenAI Codex apoyó la implementación, las derivaciones,
las interpretaciones y la validación numérica. El integrante debe revisar,
entender y poder explicar el contenido que presenta.

Ejecutar desde `notebooks/`, después de 04, en un kernel nuevo. Se leen la
partición y los hiperparámetros compartidos; no se ejecuta CV ni se cambia su
JSON. Todos son **regresores**, con predicciones sin recortar. 3.1 usa una
submuestra de 2.000 filas exclusivamente para verificar matrices completas;
3.2 vuelve a las 24.000 filas de entrenamiento y 6.000 de prueba.

In [1]:
import sys, json, gc, hashlib
from pathlib import Path
from time import perf_counter
sys.path.append("../src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from datos import cargar_taiwan
from pesos import pesos_ols, pesos_ridge, pesos_knn, pesos_arbol
from bosque import pesos_bosque, verificar
from flexibilidad import (FAMILIAS, SEMILLA, crear_modelo, diagnostico_diseno,
    espectro_centrado, traza_ridge, diagonal_knn, diagonal_arbol, diagonal_bosque,
    registrar_procedencia)

inicio_notebook = perf_counter()
hp = json.loads(Path("../resultados/hiperparametros.json").read_text())
X_train, X_test, y_train, y_test, sd = cargar_taiwan("../data/taiwan_credit.csv")
assert X_train.shape == (24000, 23) and X_test.shape == (6000, 23)
original = pd.read_csv("../data/taiwan_credit.csv")
np.testing.assert_array_equal(y_train, original.loc[X_train.index, "default"].to_numpy(float))
np.testing.assert_array_equal(y_test, original.loc[X_test.index, "default"].to_numpy(float))
assert set(X_train.index).isdisjoint(X_test.index)
SEMILLA_SUBMUESTRA = 0
pos_sub = np.random.default_rng(SEMILLA_SUBMUESTRA).choice(len(X_train), size=2000, replace=False)
X_sub, y_sub = X_train.iloc[pos_sub].copy(), y_train[pos_sub].copy()
indices_sub = X_sub.index.to_numpy().copy()
assert len(np.unique(pos_sub)) == len(X_sub) == 2000
np.testing.assert_array_equal(indices_sub, X_train.index.to_numpy()[pos_sub])
np.testing.assert_array_equal(y_sub, original.loc[indices_sub, "default"].to_numpy(float))
TOL_PREDICCION = 1e-8  # estricta, usada por verificar
TOL_TRAZA = 1e-8       # absoluta para los escalares de 3.1
diagnostico_sub = diagnostico_diseno(X_sub)
d_sub = espectro_centrado(X_sub)
duplicados_sub = int(X_sub.duplicated(keep=False).sum())
display(pd.Series({**diagnostico_sub, "filas_en_grupos_duplicados": duplicados_sub}))
display(hp)

rango_aumentado               2.400000e+01
columnas_aumentadas           2.400000e+01
tolerancia_rango              4.716782e-11
condicion_diseno              1.713301e+01
condicion_gram                2.935399e+02
filas_en_grupos_duplicados    0.000000e+00
dtype: float64

{'lambda': 316.2278, 'k': 50, 'max_depth': 4, 'min_samples_leaf': 50}

## 3.1 · Cinco trazas y sus condiciones

El suavizador de **entrenamiento** es cuadrado: $\widehat y=Sy$ en las mismas
filas y orden usados para ajustar. Una matriz de consultas contra las 24.000
filas completas no sería el suavizador de esta submuestra. La selección es una
sola llamada a `default_rng(0).choice(24000, 2000, replace=False)`, sin ordenar
después: se registran las posiciones y los índices originales en procedencia.
No se buscan nuevas submuestras hasta obtener una respuesta deseada.

**OLS.** Con $A=[\mathbf1,X]$, $S=A(A^TA)^{-1}A^T$ es una proyección. Por
ciclicidad de la traza, $\mathrm{tr}(S)=\mathrm{tr}((A^TA)^{-1}A^TA)=p+1$ si
$A$ tiene rango completo. En general, usando la pseudoinversa, la proyección
tiene traza $\mathrm{rango}(A)$; el helper compartido usa una ecuación normal
que requiere invertibilidad. Se informa rango, tolerancia SVD y condiciones
de $A$ y $A^TA$. Si falla, se detiene con diagnóstico: no se cambia el helper
ni se elige otra muestra en silencio.

**Ridge.** Al dejar libre el intercepto y centrar **esta** muestra,
$S=\mathbf1\mathbf1^T/n+X_c(X_c^TX_c+\lambda I)^{-1}X_c^T$.
Con $X_c=UDV^T$, el primer término aporta 1 y los restantes aportan
$d_j^2/(d_j^2+\lambda)$; por tanto
$\mathrm{tr}(S)=1+\sum_j d_j^2/(d_j^2+\lambda)$.
La estandarización del entrenamiento completo no centra exactamente la
submuestra. Penalizar el intercepto cambiaría este resultado (véase 1.2).

**k-NN uniforme.** $S_{ii}=\mathbf1\{i\in N_k(x_i)\}/k$. Si todos incluyen
su propia fila, $\mathrm{tr}(S)=n/k$. Se consulta `kneighbors(X_sub)` de forma
explícita: `kneighbors()` sin X excluye el propio punto. Duplicados y empates
pueden alterar qué índices se eligen, especialmente con k pequeño. La traza
realizada se calcula con los índices devueltos y se contrasta con la diagonal
de la matriz completa. La consulta adicional k+1 solo diagnostica empates en
el límite; nunca se trunca para reemplazar la selección de k.

**Árbol.** Sin pesos de muestra, una hoja ocupada por $n_L$ filas tiene
$S_{ii}=1/n_L$ en esas filas: contribuye $n_L/n_L=1$. Así, la traza es el
número de hojas de entrenamiento, aunque la partición haya dependido de y.

**Bosque.** Con `bootstrap=False`, $S$ promedia las matrices de los árboles;
$S_{ii}=T^{-1}\sum_t1/n_{L_t(x_i)}$. Se calcula esta diagonal a partir de las
pertenencias reales, se suma y se contrasta con `np.trace(S)`. Como control
adicional, coincide con el promedio de los números de hojas. No se usan estas
ponderaciones en un bosque bootstrap, que necesita multiplicidades.